# 🔬 결합 피처 군집 분석 (Combined Feature Clustering)

이 노트북에서는 **성능이 좋았던 여러 피처들을 결합**하여 고차원 특징벡터를 만들고, 이를 통해 더 정확한 군집 분석을 수행합니다.

## 📋 목차
1. **데이터 로드**: combined 폴더 제외
2. **다양한 피처 추출**: 성능 비교
3. **고차원 특징벡터 생성**: 상위 성능 피처들 결합
4. **결합 피처 군집 분석**: PCA + K-means
5. **성능 비교**: 단일 피처 vs 결합 피처


In [ ]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# 공통 유틸리티
from utils import setup_plotting, get_data_dir, get_state_mapping, get_state_names

# 머신러닝
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, davies_bouldin_score
)
import seaborn as sns

# 프로젝트 모듈
from app.ml.features.extractor import AudioFeatureExtractor, AudioConfig

# 시각화 설정
setup_plotting()

print(f"✅ 라이브러리 로드 완료!")


---
## 1. 데이터 로드 (combined 폴더 제외)


In [ ]:
# ============================================================
# 데이터 경로 수집 (combined 폴더 제외)
# ============================================================

data_dir = get_data_dir()
augmented_dir = data_dir / 'augmented'

# 파일 경로와 레이블 수집
all_files = []
all_states = []

# 상태 매핑
state_mapping = get_state_mapping()
state_names = get_state_names()

print("📂 데이터 로드 중... (combined 폴더 제외)")

# 원본 데이터 수집
for state_dir in sorted(data_dir.iterdir()):
    if not state_dir.is_dir() or state_dir.name == 'augmented':
        continue
    
    state_name = state_dir.name
    state_idx = state_mapping.get(state_name, -1)
    
    if state_idx == -1:
        continue
    
    for problem_dir in sorted(state_dir.iterdir()):
        if not problem_dir.is_dir():
            continue
        
        problem_name = problem_dir.name
        
        # ⚠️ combined 폴더 제외
        if problem_name == 'combined':
            continue
        
        # WAV 파일 수집
        wav_files = list(problem_dir.glob('*.wav'))
        for f in wav_files:
            all_files.append(f)
            all_states.append(state_idx)
        
        # 하위 폴더 확인 (combined 제외)
        for sub_dir in problem_dir.iterdir():
            if sub_dir.is_dir() and sub_dir.name != 'combined':
                sub_files = list(sub_dir.glob('*.wav'))
                for f in sub_files:
                    all_files.append(f)
                    all_states.append(state_idx)

original_count = len(all_files)
print(f"   원본 샘플: {original_count}개")

# 증강 데이터 수집
if augmented_dir.exists():
    for state_dir in sorted(augmented_dir.iterdir()):
        if not state_dir.is_dir():
            continue
        
        state_name = state_dir.name
        state_idx = state_mapping.get(state_name, -1)
        
        if state_idx == -1:
            continue
        
        for problem_dir in sorted(state_dir.iterdir()):
            if not problem_dir.is_dir():
                continue
            
            problem_name = problem_dir.name
            
            # ⚠️ combined 폴더 제외
            if problem_name == 'combined':
                continue
            
            # 증강 WAV 파일 수집
            aug_files = list(problem_dir.glob('*.wav'))
            for f in aug_files:
                all_files.append(f)
                all_states.append(state_idx)
            
            # 하위 폴더 확인 (combined 제외)
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and sub_dir.name != 'combined':
                    sub_files = list(sub_dir.glob('*.wav'))
                    for f in sub_files:
                        all_files.append(f)
                        all_states.append(state_idx)

augmented_count = len(all_files) - original_count
print(f"   증강 샘플: {augmented_count}개")

print("\n" + "=" * 50)
print(f"📊 총 데이터: {len(all_files)}개 (combined 제외)")
print("=" * 50)

# 상태별 분포 확인
state_counts = Counter(all_states)
print("\n📊 상태별 분포:")
for idx, name in enumerate(state_names):
    print(f"  [{idx}] {name}: {state_counts[idx]}개")


---
## 2. 피처 추출 및 성능 평가


In [ ]:
# ============================================================
# 피처 추출기 초기화
# ============================================================

audio_config = AudioConfig(
    sample_rate=22050,
    duration=5.0,
    n_mels=128,
    n_mfcc=40,
    n_fft=2048,
    hop_length=512
)

feature_extractor = AudioFeatureExtractor(config=audio_config)
print("✅ 피처 추출기 초기화 완료!")


In [ ]:
# ============================================================
# 다양한 피처 추출 함수 정의
# ============================================================

def extract_multiple_features(file_path, feature_extractor, sr=22050):
    """
    다양한 피처를 추출하는 함수
    
    Returns:
        dict: 각 피처 종류별 추출된 값 (flatten된 벡터)
    """
    # 오디오 로드
    y, sr = feature_extractor.load_audio(str(file_path))
    
    features = {}
    
    # 1. Mel Spectrogram (기본)
    mel_spec = feature_extractor.extract_mel_spectrogram(y, sr)
    features['mel_spectrogram'] = mel_spec.flatten()
    
    # 2. MFCC (Delta 없이)
    mfcc = feature_extractor.extract_mfcc(y, sr, include_delta=False)
    features['mfcc'] = mfcc.flatten()
    
    # 3. MFCC + Delta + Delta2
    mfcc_delta = feature_extractor.extract_mfcc(y, sr, include_delta=True)
    features['mfcc_delta'] = mfcc_delta.flatten()
    
    # 4. Chroma
    chroma = feature_extractor.extract_chroma(y, sr)
    features['chroma'] = chroma.flatten()
    
    # 5. Spectral Contrast
    contrast = feature_extractor.extract_spectral_contrast(y, sr)
    features['spectral_contrast'] = contrast.flatten()
    
    # 6. Spectral Features (통계적 특성)
    spectral = feature_extractor.extract_spectral_features(y, sr)
    spectral_combined = np.concatenate([
        spectral['spectral_centroid'].flatten(),
        spectral['spectral_bandwidth'].flatten(),
        spectral['spectral_rolloff'].flatten(),
        spectral['zero_crossing_rate'].flatten(),
        spectral['rms'].flatten()
    ])
    features['spectral_features'] = spectral_combined
    
    # 7. 통계 기반 피처 (평균, 표준편차, 최대, 최소)
    mel_stats = np.concatenate([
        mel_spec.mean(axis=1),  # 각 Mel 밴드의 평균
        mel_spec.std(axis=1),   # 각 Mel 밴드의 표준편차
        mel_spec.max(axis=1),   # 각 Mel 밴드의 최대값
        mel_spec.min(axis=1)    # 각 Mel 밴드의 최소값
    ])
    features['mel_stats'] = mel_stats
    
    mfcc_stats = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        mfcc.max(axis=1),
        mfcc.min(axis=1)
    ])
    features['mfcc_stats'] = mfcc_stats
    
    # 8. Chroma 통계
    chroma_stats = np.concatenate([
        chroma.mean(axis=1),
        chroma.std(axis=1),
        chroma.max(axis=1),
        chroma.min(axis=1)
    ])
    features['chroma_stats'] = chroma_stats
    
    return features

print("✅ 다양한 피처 추출 함수 정의 완료!")


In [ ]:
# ============================================================
# 모든 피처 추출 및 성능 평가
# ============================================================

print("🔄 다양한 피처 추출 중... (시간이 걸릴 수 있습니다)")

# 피처 저장용 딕셔너리
feature_dict = {
    'mel_spectrogram': [],
    'mfcc': [],
    'mfcc_delta': [],
    'chroma': [],
    'spectral_contrast': [],
    'spectral_features': [],
    'mel_stats': [],
    'mfcc_stats': [],
    'chroma_stats': []
}

# 샘플링 (전체 데이터가 너무 많으면 일부만 사용)
MAX_SAMPLES = min(1500, len(all_files))  # 최대 1500개 샘플
sample_indices = np.random.choice(len(all_files), MAX_SAMPLES, replace=False)
sampled_files = [all_files[i] for i in sample_indices]
sampled_states = [all_states[i] for i in sample_indices]

print(f"   샘플링: {len(all_files)}개 중 {MAX_SAMPLES}개 사용")

for file_path in tqdm(sampled_files, desc="다양한 피처 추출"):
    try:
        features = extract_multiple_features(file_path, feature_extractor)
        for key in feature_dict.keys():
            feature_dict[key].append(features[key])
    except Exception as e:
        print(f"Error: {file_path.name} - {e}")
        # 오류 시 0으로 채움
        for key in feature_dict.keys():
            if len(feature_dict[key]) > 0:
                feature_dict[key].append(np.zeros_like(feature_dict[key][-1]))

# NumPy 배열로 변환
for key in feature_dict.keys():
    feature_dict[key] = np.array(feature_dict[key])

y_sampled = np.array(sampled_states)

print(f"\n✅ 다양한 피처 추출 완료!")
print("\n📊 피처별 차원:")
for key, value in feature_dict.items():
    print(f"   {key}: {value.shape}")


In [ ]:
# ============================================================
# 피처별 클러스터링 성능 평가
# ============================================================

def evaluate_clustering_performance(feature_dict, y_labels, state_names):
    """각 피처에 대해 K-means 클러스터링 성능 평가"""
    
    results = []
    
    for feature_name, X_feat in feature_dict.items():
        # 정규화
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_feat)
        
        # K-means 클러스터링
        kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
        cluster_labels = kmeans.fit_predict(X_scaled)
        
        # 성능 평가
        silhouette = silhouette_score(X_scaled, cluster_labels)
        ari = adjusted_rand_score(y_labels, cluster_labels)
        nmi = normalized_mutual_info_score(y_labels, cluster_labels)
        db_score = davies_bouldin_score(X_scaled, cluster_labels)
        
        results.append({
            'Feature': feature_name,
            'Silhouette': silhouette,
            'ARI': ari,
            'NMI': nmi,
            'DB_Score': db_score,  # 낮을수록 좋음
            'Dimension': X_feat.shape[1]
        })
    
    return pd.DataFrame(results)

# 클러스터링 성능 평가
print("🔄 피처별 클러스터링 성능 평가 중...")
df_results = evaluate_clustering_performance(feature_dict, y_sampled, state_names)

# 결과 정렬 (ARI 기준)
df_results_sorted = df_results.sort_values('ARI', ascending=False)

print("\n" + "=" * 80)
print("📊 피처별 클러스터링 성능 비교 (K-means, k=3)")
print("=" * 80)
print(df_results_sorted.to_string(index=False))
print("\n💡 ARI (Adjusted Rand Index)가 높을수록 실제 레이블과 일치도가 높습니다.")


---
## 3. 고차원 특징벡터 생성 (상위 성능 피처 결합)


In [ ]:
# ============================================================
# 상위 성능 피처 선택 및 결합
# ============================================================

# ARI 기준으로 상위 N개 피처 선택
TOP_N = 5  # 상위 5개 피처 결합
top_features = df_results_sorted.head(TOP_N)['Feature'].tolist()

print(f"🏆 상위 {TOP_N}개 성능 피처 선택:")
print("=" * 60)
for idx, feat in enumerate(top_features, 1):
    feat_info = df_results_sorted[df_results_sorted['Feature'] == feat].iloc[0]
    print(f"  {idx}. {feat}")
    print(f"     - ARI: {feat_info['ARI']:.4f}")
    print(f"     - Silhouette: {feat_info['Silhouette']:.4f}")
    print(f"     - 차원: {int(feat_info['Dimension'])}")

# 선택된 피처들의 특징벡터 결합
print(f"\n🔄 고차원 특징벡터 생성 중...")

combined_features_list = []

for i in range(len(sampled_files)):
    # 각 샘플에 대해 상위 피처들을 결합
    sample_features = []
    
    for feat_name in top_features:
        feat_vector = feature_dict[feat_name][i]
        sample_features.append(feat_vector)
    
    # 결합 (concatenate)
    combined_vector = np.concatenate(sample_features)
    combined_features_list.append(combined_vector)

# NumPy 배열로 변환
X_combined = np.array(combined_features_list)

print(f"\n✅ 고차원 특징벡터 생성 완료!")
print(f"   결합 전 차원: {[feature_dict[f].shape[1] for f in top_features]}")
print(f"   결합 후 차원: {X_combined.shape}")
print(f"   총 피처 수: {X_combined.shape[1]}개")


---
## 4. 결합 피처 군집 분석


In [ ]:
# ============================================================
# 결합 피처 정규화 및 PCA
# ============================================================

# 정규화 (고차원 데이터이므로 StandardScaler 사용)
scaler_combined = StandardScaler()
X_combined_scaled = scaler_combined.fit_transform(X_combined)

print(f"✅ 결합 피처 정규화 완료!")

# 2D PCA
print("🔄 PCA 수행 중...")
pca_2d = PCA(n_components=2, random_state=42)
X_combined_pca_2d = pca_2d.fit_transform(X_combined_scaled)

print(f"\n✅ PCA (2D) 완료!")
print(f"   설명된 분산 비율: {pca_2d.explained_variance_ratio_.sum():.4f}")
print(f"   PC1: {pca_2d.explained_variance_ratio_[0]:.4f} ({pca_2d.explained_variance_ratio_[0]:.2%})")
print(f"   PC2: {pca_2d.explained_variance_ratio_[1]:.4f} ({pca_2d.explained_variance_ratio_[1]:.2%})")

# 3D PCA
pca_3d = PCA(n_components=3, random_state=42)
X_combined_pca_3d = pca_3d.fit_transform(X_combined_scaled)

print(f"\n✅ PCA (3D) 완료!")
print(f"   설명된 분산 비율: {pca_3d.explained_variance_ratio_.sum():.4f}")


In [ ]:
# ============================================================
# 결합 피처 PCA 2D 시각화
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
markers = ['o', 's', '^']

for idx, (name, color, marker) in enumerate(zip(state_names, colors, markers)):
    mask = y_sampled == idx
    ax.scatter(
        X_combined_pca_2d[mask, 0], 
        X_combined_pca_2d[mask, 1], 
        c=color, 
        marker=marker,
        label=f'{name} (n={mask.sum()})',
        alpha=0.6,
        s=50
    )

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
ax.set_title(f'🎯 결합 피처 PCA 2D 시각화 (상위 {TOP_N}개 피처 결합)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 해석:")
print("  • 결합 피처를 사용하면 클래스 간 분리가 더 명확해질 수 있습니다.")
print("  • 같은 색상(상태)의 점들이 더 잘 모여있으면 분류가 쉬움")


In [ ]:
# ============================================================
# 결합 피처 PCA 3D 시각화
# ============================================================

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

for idx, (name, color, marker) in enumerate(zip(state_names, colors, markers)):
    mask = y_sampled == idx
    ax.scatter(
        X_combined_pca_3d[mask, 0], 
        X_combined_pca_3d[mask, 1], 
        X_combined_pca_3d[mask, 2],
        c=color, 
        marker=marker,
        label=f'{name} (n={mask.sum()})',
        alpha=0.6,
        s=50
    )

ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.2%})')
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.2%})')
ax.set_title(f'🎯 결합 피처 PCA 3D 시각화 (상위 {TOP_N}개 피처 결합)', fontsize=14, fontweight='bold')
ax.legend(loc='best')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 결합 피처 K-Means 클러스터링
# ============================================================

print("🔄 K-Means 클러스터링 수행 중...")

# K=3으로 클러스터링 (3개 상태에 맞춤)
kmeans_combined = KMeans(n_clusters=3, random_state=42, n_init='auto')
cluster_labels_combined = kmeans_combined.fit_predict(X_combined_scaled)

print(f"\n✅ K-Means 클러스터링 완료!")

# 클러스터별 분포
cluster_counts = Counter(cluster_labels_combined)
print("\n📊 클러스터별 샘플 수:")
for cluster_id in sorted(cluster_counts.keys()):
    print(f"  Cluster {cluster_id}: {cluster_counts[cluster_id]}개")


In [ ]:
# ============================================================
# 결합 피처 클러스터링 결과 시각화
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 좌측: 실제 레이블
for idx, (name, color, marker) in enumerate(zip(state_names, colors, markers)):
    mask = y_sampled == idx
    axes[0].scatter(
        X_combined_pca_2d[mask, 0], 
        X_combined_pca_2d[mask, 1], 
        c=color, 
        marker=marker,
        label=f'{name}',
        alpha=0.6,
        s=50
    )
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('✅ 실제 레이블 (Ground Truth)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 우측: K-Means 클러스터
cluster_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for cluster_id in range(3):
    mask = cluster_labels_combined == cluster_id
    axes[1].scatter(
        X_combined_pca_2d[mask, 0], 
        X_combined_pca_2d[mask, 1], 
        c=cluster_colors[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        s=50
    )

# 클러스터 중심 표시
centers_pca = pca_2d.transform(kmeans_combined.cluster_centers_)
axes[1].scatter(
    centers_pca[:, 0], centers_pca[:, 1],
    c='black', marker='X', s=200, edgecolors='white', linewidths=2,
    label='Centroids'
)

axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title(f'🔵 K-Means 클러스터링 결과 (결합 피처)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'📊 실제 레이블 vs K-Means 클러스터링 (결합 피처: 상위 {TOP_N}개)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 결합 피처 클러스터링 성능 평가
# ============================================================

# 1. Silhouette Score (군집 응집도)
silhouette_combined = silhouette_score(X_combined_scaled, cluster_labels_combined)

# 2. Adjusted Rand Index (실제 레이블과의 일치도)
ari_combined = adjusted_rand_score(y_sampled, cluster_labels_combined)

# 3. Normalized Mutual Information
nmi_combined = normalized_mutual_info_score(y_sampled, cluster_labels_combined)

# 4. Davies-Bouldin Score (낮을수록 좋음)
db_combined = davies_bouldin_score(X_combined_scaled, cluster_labels_combined)

print("=" * 70)
print("📊 결합 피처 클러스터링 성능 평가")
print("=" * 70)
print(f"\n1️⃣ Silhouette Score: {silhouette_combined:.4f}")
print("   (-1 ~ 1, 높을수록 군집이 잘 분리됨)")
print(f"\n2️⃣ Adjusted Rand Index (ARI): {ari_combined:.4f}")
print("   (0 ~ 1, 높을수록 실제 레이블과 일치)")
print(f"\n3️⃣ Normalized Mutual Information (NMI): {nmi_combined:.4f}")
print("   (0 ~ 1, 높을수록 실제 레이블과 일치)")
print(f"\n4️⃣ Davies-Bouldin Score: {db_combined:.4f}")
print("   (0 이상, 낮을수록 군집이 잘 분리됨)")


In [ ]:
# ============================================================
# 클러스터-레이블 매칭 분석
# ============================================================

# 혼동 행렬: 클러스터 vs 실제 레이블
contingency = pd.crosstab(
    pd.Series(cluster_labels_combined, name='Cluster'),
    pd.Series(y_sampled, name='State').map(lambda x: state_names[x])
)

print("\n📊 클러스터-레이블 교차표:")
print(contingency)

# 히트맵 시각화
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    contingency, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    ax=ax
)
ax.set_title(f'🔍 클러스터 vs 실제 레이블 분포 (결합 피처: 상위 {TOP_N}개)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('실제 상태 (State)', fontsize=12)
ax.set_ylabel('K-Means 클러스터', fontsize=12)
plt.tight_layout()
plt.show()

# 각 클러스터의 주요 레이블 분석
print("\n💡 클러스터별 주요 구성:")
for cluster_id in range(3):
    cluster_mask = cluster_labels_combined == cluster_id
    states_in_cluster = y_sampled[cluster_mask]
    state_dist = Counter(states_in_cluster)
    
    total = sum(state_dist.values())
    print(f"\n  Cluster {cluster_id}:")
    for state_idx, count in sorted(state_dist.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        print(f"    {state_names[state_idx]}: {count}개 ({pct:.1f}%)")


---
## 5. 성능 비교: 단일 피처 vs 결합 피처


In [ ]:
# ============================================================
# 단일 최고 성능 피처와 결합 피처 비교
# ============================================================

# 최고 성능 단일 피처
best_single_feature = df_results_sorted.iloc[0]['Feature']
best_single_ari = df_results_sorted.iloc[0]['ARI']
best_single_silhouette = df_results_sorted.iloc[0]['Silhouette']
best_single_nmi = df_results_sorted.iloc[0]['NMI']

# 비교 데이터프레임 생성
comparison_data = {
    '방법': [
        f'단일 피처: {best_single_feature}',
        f'결합 피처: 상위 {TOP_N}개'
    ],
    'ARI': [best_single_ari, ari_combined],
    'Silhouette': [best_single_silhouette, silhouette_combined],
    'NMI': [best_single_nmi, nmi_combined],
    '차원': [
        int(df_results_sorted.iloc[0]['Dimension']),
        X_combined.shape[1]
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("=" * 80)
print("📊 단일 피처 vs 결합 피처 성능 비교")
print("=" * 80)
print("\n" + comparison_df.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['ARI', 'Silhouette', 'NMI']
metric_names = ['Adjusted Rand Index', 'Silhouette Score', 'Normalized Mutual Info']

for idx, (metric, metric_name) in enumerate(zip(metrics, metric_names)):
    colors = ['#4ECDC4', '#FF6B6B']
    bars = axes[idx].bar(comparison_df['방법'], comparison_df[metric], color=colors, alpha=0.7)
    
    # 값 표시
    for bar, val in zip(bars, comparison_df[metric]):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                      f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    axes[idx].set_ylabel(metric_name, fontsize=11)
    axes[idx].set_title(f'📈 {metric_name}', fontsize=12, fontweight='bold')
    axes[idx].set_ylim(0, max(comparison_df[metric]) * 1.2)
    axes[idx].grid(axis='y', alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=15)

plt.suptitle('🔍 단일 피처 vs 결합 피처 성능 비교', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 개선율 계산
ari_improvement = ((ari_combined - best_single_ari) / best_single_ari) * 100
silhouette_improvement = ((silhouette_combined - best_single_silhouette) / best_single_silhouette) * 100
nmi_improvement = ((nmi_combined - best_single_nmi) / best_single_nmi) * 100

print(f"\n💡 결합 피처의 성능 개선:")
print(f"   • ARI: {ari_improvement:+.2f}%")
print(f"   • Silhouette: {silhouette_improvement:+.2f}%")
print(f"   • NMI: {nmi_improvement:+.2f}%")

if ari_combined > best_single_ari:
    print(f"\n✅ 결합 피처가 단일 피처보다 {ari_combined - best_single_ari:.4f} 높은 ARI를 보였습니다!")
    print(f"   → 여러 피처를 결합하면 군집 분석 성능이 향상될 수 있습니다.")
else:
    print(f"\nℹ️  단일 피처가 더 나은 성능을 보였습니다.")
    print(f"   → 차원의 저주나 피처 간 상관관계로 인해 결합이 항상 유리한 것은 아닙니다.")


---
## 6. 최종 결과 요약


In [ ]:
# ============================================================
# 전체 결과 요약
# ============================================================

print("=" * 80)
print("🎉 결합 피처 군집 분석 완료!")
print("=" * 80)

print(f"""
📊 분석 요약:

┌─────────────────────────────────────────────────────────────────┐
│ 데이터 정보                                                      │
│   • 총 데이터: {len(all_files)}개 (combined 폴더 제외)         │
│   • 분석 샘플: {MAX_SAMPLES}개                                  │
│   • 상태별 분포:                                                │
│     - braking: {state_counts[0]}개                              │
│     - idle: {state_counts[1]}개                                 │
│     - startup: {state_counts[2]}개                             │
├─────────────────────────────────────────────────────────────────┤
│ 결합 피처 정보                                                  │
│   • 결합된 피처 수: {TOP_N}개                                   │
│   • 결합 피처 목록: {', '.join(top_features)}                   │
│   • 고차원 특징벡터 차원: {X_combined.shape[1]}개               │
├─────────────────────────────────────────────────────────────────┤
│ 결합 피처 군집 분석 결과                                        │
│   • Silhouette Score: {silhouette_combined:.4f}                │
│   • Adjusted Rand Index: {ari_combined:.4f}                   │
│   • Normalized Mutual Info: {nmi_combined:.4f}                 │
│   • Davies-Bouldin Score: {db_combined:.4f}                    │
├─────────────────────────────────────────────────────────────────┤
│ 성능 비교                                                       │
│   • 단일 최고 피처 ({best_single_feature}):                    │
│     - ARI: {best_single_ari:.4f}                               │
│   • 결합 피처 (상위 {TOP_N}개):                                │
│     - ARI: {ari_combined:.4f}                                  │
│     - 개선율: {ari_improvement:+.2f}%                          │
└─────────────────────────────────────────────────────────────────┘

💡 주요 특징:
  • 성능이 좋은 여러 피처를 결합하여 고차원 특징벡터 생성
  • 결합 피처를 사용한 군집 분석으로 더 정확한 클러스터링 가능
  • 단일 피처 대비 성능 개선 여부 확인
  • PCA를 통한 시각화로 클러스터 분리도 확인
""")
print("=" * 80)
